In [ ]:
import os, warnings
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from src.plotting import plot_autocorrelation_analysis, plot_subject_timeseries_heatmap

warnings.simplefilter("ignore", UserWarning)

In [ ]:
dataset = 'HCP-YA'
# dataset = 'HCP-D'
# dataset = 'WAND'
# dataset = 'PNC'
# dataset = 'UCLA'
# dataset = 'NKI'
data_loader = DataLoader(dataset=dataset)

# fMRI data

In [ ]:
seg = 'Schaefer4007'

if dataset == 'HCP-YA':
    run_meg = True
    if run_meg:
        # subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_meg.txt'), dtype=str)
        subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_subsample.txt'), dtype=str)
    else:
        subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_7t.txt'), dtype=str)
    sessions = ['01',]
    tasks = ['rfMRIREST1LR', 'rfMRIREST1RL', 'rfMRIREST2LR', 'rfMRIREST2RL',
             'tfMRIWMLR', 'tfMRIWMRL',
             'tfMRIEMOTIONLR', 'tfMRIEMOTIONRL',
             'tfMRIGAMBLINGLR', 'tfMRIGAMBLINGRL',
             'tfMRILANGUAGELR', 'tfMRILANGUAGERL',
             'tfMRIRELATIONALLR', 'tfMRIRELATIONALRL',
             'tfMRISOCIALLR', 'tfMRISOCIALRL',
             'tfMRIMOTORLR', 'tfMRIMOTORRL']
    tasks_7t = ['rfMRIREST17TPA', 'rfMRIREST27TAP', 'rfMRIREST37TPA', 'rfMRIREST47TAP',
                'tfMRIMOVIE17TAP', 'tfMRIMOVIE27TPA', 'tfMRIMOVIE37TPA', 'tfMRIMOVIE47TAP']
    tasks_meg = ['3Restin', '4Restin', '5Restin', '6Wrkmem', '7Wrkmem']
    space = 'fsLR'
    nuisance_regression = False
    task_regression = True
elif dataset == 'HCP-D':
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = [subj_id.replace('_', '') for subj_id in subject_ids]
    sessions = ['01',]
    tasks = ['rfMRIREST1PA', 'rfMRIREST2PA',
             'tfMRICARITPA', 'tfMRIEMOTIONPA', 'tfMRIGUESSINGPA']
    space = 'fsLR'
    nuisance_regression = False
    task_regression = True
elif dataset == 'WAND':
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_bids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['03',]
    tasks = ['rest', 'reversallearning']
    space = 'MNI152NLin2009cAsym'
    nuisance_regression = False
    task_regression = False
elif dataset == 'PNC':
    seg = 'Schaefer2018p400n17'
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['PNC1',]
    tasks = ['rest', 'frac2back']
    space = 'MNI152NLin6ASym'
    nuisance_regression = False
    task_regression = True
elif dataset == 'UCLA':
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_not1_hc.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = [None,]
    tasks = ['rest', 'bart', 'stopsignal'] # 'taskswitch']
    space = 'MNI152NLin2009cAsym'
    nuisance_regression = False
    task_regression = True
elif dataset == 'NKI':
    seg = 'Schaefer2018p400n17'
    subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
    subject_ids = [i.split('-')[1] for i in subject_ids]
    sessions = ['BAS1',]
    tasks = ['rest', 'CHECKERBOARD']  # 'BREATHHOLD'
    space = 'MNI152NLin6ASym'
    nuisance_regression = False
    task_regression = True
# subject_ids = subject_ids[:100]
# subject_ids = subject_ids[:50]
subject_ids

In [ ]:
# Register 3T rest scans
scans = data_loader.register_scans(
    subject_ids=subject_ids,
    sessions=sessions,
    tasks=tasks,
    space=space,
    seg=seg,
    modality='MRI3T'
)

if dataset == 'HCP-YA':
    scans_7t = data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=sessions,
        tasks=tasks_7t,
        space=space,
        seg=seg,
        modality='MRI7T'
    )
    
if dataset == 'HCP-YA' and run_meg:
    scans = data_loader.register_scans(
        subject_ids=subject_ids,
        sessions=sessions,
        tasks=tasks_meg,
        space=space,
        seg=seg,
        modality='MEG'
    )

In [ ]:
_ = data_loader.validate_scan_registry(remove_incomplete_subjects=True)

In [ ]:
# Load the data
fmri_data = data_loader.load_batch(
    nuisance_regression=nuisance_regression,
    standardize=False,
    verbose=True
)

print(f"Loaded {len(fmri_data)} scans")
print(fmri_data.keys())

In [ ]:
data_loader.export_scan_registry(f'scan_registry_{dataset}.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['sub'].unique()
n_subs = len(subject_ids)
print(n_subs)

In [ ]:
subject_ids

## Process fMRI timescales

In [ ]:
fmri_data[list(fmri_data.keys())[0]].shape

In [ ]:
fmri_data[list(fmri_data.keys())[-1]].shape

In [ ]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=fmri_data, verbose=True)
if task_regression:
    timescales_processor.regress_task_structure()

### timeseries metrics (not timescales)

In [ ]:
run_cell = True

In [ ]:
if run_cell:
    for method in ['tsnr', 'var']:
        timescales_processor.compute_timeseries_measures(method=method, force_recompute=False, verbose=True)

        if task_regression:
            timescales_processor.compute_timeseries_measures(method=method, use_residuals=True, force_recompute=False, verbose=True)

### Timescale measures

In [ ]:
timescale_measures = [
    'zero_crossing',
    # 'auc_zero_crossing',
    # 'tau_optimized',
    # 'B_optimized',
]

In [ ]:
timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, force_recompute=False, verbose=True)
for timescale_measure in timescale_measures:
    timescales_processor.compute_timescale_measures(method=timescale_measure, force_recompute=False, verbose=True)
    
autocorr_dict_raw = timescales_processor.autocorr_dict.copy()
timescales_dict_raw = timescales_processor.timescales_dict.copy()

In [ ]:
if task_regression:
    timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, force_recompute=False, verbose=True)
    for timescale_measure in timescale_measures:
        timescales_processor.compute_timescale_measures(method=timescale_measure, use_residuals=True, suffix='resids', force_recompute=False, verbose=True)
        
    autocorr_dict_resid = timescales_processor.autocorr_dict.copy()
    timescales_dict_resid = timescales_processor.timescales_dict.copy()

In [ ]:
# for idx in scan_registry.index:
#     scan_string = scan_registry.loc[idx, 'scan_string']
#     print(timescales_dict_raw[scan_string][:5])

In [ ]:
# design_matrix = timescales_processor.design_matrices['sub-109123_modality-MRI3T_tr-0.72_condition-task_scan-tfMRIWMLR_atlas-Schaefer4007']
# plt.plot(design_matrix['trial'])

In [ ]:
# plt.plot(compute_acf(design_matrix['trial'].values)[0, :20])

## Generate plots

In [ ]:
run_cell = True

In [ ]:
# if run_cell:
#     unique_scans = list(scan_registry.loc[scan_registry.loc[:, 'condition'] == 'task', 'scan'].unique())
#     fig, axes = plt.subplots(1, len(unique_scans), figsize=(30, 2))

#     for idx in scan_registry.index:
#         scan_string = scan_registry.loc[idx, 'scan_string']
#         this_condition = scan_registry.loc[idx, 'condition']
#         this_scan = scan_registry.loc[idx, 'scan']
        
#         if this_condition == 'task':
#             if this_scan in unique_scans:
#                 this_ax = unique_scans.index(this_scan)
#                 axes[this_ax].plot(timescales_processor.task_events_acf[scan_string][0, :25])
                
#     fig.tight_layout()

In [ ]:
if run_cell:
    for idx in scan_registry.index[:10]:
        # if idx & 10 == 0 and scan_registry.loc[idx, 'condition'] == 'task':
        if scan_registry.loc[idx, 'condition'] == 'task':
            scan_string = scan_registry.loc[idx, 'scan_string']
            tr = scan_registry.loc[idx, 'tr']
            print(idx, scan_string)
        
            raw_data = timescales_processor.timeseries_data[scan_string]
            residual_data = timescales_processor.timeseries_residuals[scan_string]
            design_matrix = timescales_processor.design_matrices[scan_string]

            autocorr_raw = autocorr_dict_raw[scan_string]
            autocorr_resid = autocorr_dict_resid[scan_string]
            timescales_raw = timescales_dict_raw[scan_string]
            timescales_resid = timescales_dict_resid[scan_string]

            plot_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'plots', 'plot_subject_timeseries_heatmap')
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)


            fig, axes = plot_subject_timeseries_heatmap(raw_data, residual_data, design_matrix, scan_string,
                                                        figsize=(15, 10), dpi=100,
                                                        save_path=os.path.join(plot_dir, f'{scan_string}.png')
                                                        )
            
            plot_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'plots', 'plot_autocorrelation_analysis')
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)

            fig = plot_autocorrelation_analysis(autocorr_raw, autocorr_resid, timescales_raw, timescales_resid, 
                                                tr, max_lag=20, figsize=(15, 4), dpi=100, 
                                                save_path=os.path.join(plot_dir, f'{scan_string}.png')
                                                )
            plt.close('all')